In [1]:
import re
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display, Markdown
from numpyro import distributions as dist

from emu_renewal.inputs import get_worldbank_national_pop, get_income_group, get_fb_visited_mobility, \
    get_fb_singletile_mobility, process_raw_google_mobility, get_google_mobility, get_country_pop, \
    get_undesa_national_pop, get_country_vacc_data
from emu_renewal.document import get_func_blurb
from emu_renewal.indicators import get_deaths_target, get_cases_target, filter_seroprev, get_owid_hosps, \
    get_all_seroprev, get_seroprev_pooled_totals, get_var_target, extract_specific_var, get_country_vars, \
    get_alpha_info, get_delta_info, get_specific_var_props, get_ba2_target, get_ba5_target, \
    get_ba2_info, get_ba5_info
from emu_renewal.run import find_run_start_time, find_run_end_time, get_hosp_target, get_seroprev_target, \
    get_mobility_provider, run_calibration
from emu_renewal.renew import MultiStrainModel
from emu_renewal.distributions import GammaDens
from emu_renewal.process import _get_cos_curve_at_x
from emu_renewal.mobility import NoMobilityProvider, WeightedExpMobilityProvider, SingleSeriesExpMobilityProvider
from emu_renewal.calibration import custom_init, StandardCalib

In [2]:
txt = "## Modelled population size\n"
txt += get_func_blurb(get_country_pop)
txt += get_func_blurb(get_worldbank_national_pop)
txt += get_func_blurb(get_undesa_national_pop)
txt += "\n\n## Analysis period\n"
txt += get_func_blurb(find_run_start_time)
txt += get_func_blurb(get_country_vacc_data) + "\n\n"
txt += get_func_blurb(find_run_end_time)
txt += "\n\n## Calibration targets\n"
txt += "\n\n### WHO indicators\n"
txt += get_func_blurb(get_deaths_target)
txt += get_func_blurb(get_cases_target)
txt += "\n\n### Hospitalisations\n"
txt += get_func_blurb(get_owid_hosps)
txt += get_func_blurb(get_hosp_target) + "\n\n"
txt += "\n\n### Seroprevalence\n"
txt += get_func_blurb(get_all_seroprev)
txt += get_func_blurb(get_income_group)
txt += get_func_blurb(filter_seroprev)
txt += get_func_blurb(get_seroprev_pooled_totals) + "\n\n"
txt += get_func_blurb(get_seroprev_target) + "\n\n"
txt += "\n\n### Variants\n\n"
txt += "#### Data extraction\n"
txt += get_func_blurb(get_country_vars)
txt += get_func_blurb(extract_specific_var)
txt += get_func_blurb(get_specific_var_props) + "\n\n"
txt += get_func_blurb(get_var_target)
txt += "\n\n#### Alpha\n"
txt += get_func_blurb(get_alpha_info)
txt += "\n\n#### Delta\n"
txt += get_func_blurb(get_delta_info)
txt += "\n\n#### Omicron BA.2\n"
txt += get_func_blurb(get_ba2_target)
txt += get_func_blurb(get_ba2_info)
txt += "\n\n#### Omicron BA.5\n"
txt += get_func_blurb(get_ba5_target)
txt += get_func_blurb(get_ba5_info)
txt += "\n\n## Mobility\n"
txt += get_func_blurb(get_mobility_provider)
txt += "\n\n### No mobility\n"
no_mob_provider = NoMobilityProvider()
txt += get_func_blurb(NoMobilityProvider.__init__)
txt += "\n\n### Google mobility\n"
n_domains = 6
dummy_g_mob = pd.DataFrame(1.0, index=range(10), columns=range(n_domains))
dummy_g_priors = {"mob_weights": dist.Uniform(np.zeros(n_domains), np.ones(n_domains)), "mob_exp": None}
weighted_exp_mob_provider = WeightedExpMobilityProvider(dummy_g_mob, dummy_g_priors)
txt += get_func_blurb(process_raw_google_mobility)
txt += get_func_blurb(get_google_mobility)
txt += get_func_blurb(weighted_exp_mob_provider.__init__)
txt += get_func_blurb(weighted_exp_mob_provider.get_parameterised_mobility)
txt += "\n\n### Facebook mobility\n"
txt += get_func_blurb(get_fb_visited_mobility)
txt += get_func_blurb(get_fb_singletile_mobility)
dummy_fb_mob = pd.Series(1.0, index=range(10))
dummy_fb_priors = {"mob_exp": None}
single_series_mob_provider = SingleSeriesExpMobilityProvider(dummy_fb_mob, dummy_fb_priors)
txt += get_func_blurb(single_series_mob_provider.__init__)
txt += get_func_blurb(single_series_mob_provider.get_parameterised_mobility)
dummy_start = datetime(2020, 1, 1)
dummy_end = datetime(2021, 12, 31)
dummy_model = MultiStrainModel(1.0, dummy_start, dummy_end, [""], dummy_start, no_mob_provider, False)
txt += "\n\n## Variable process\n"
txt += get_func_blurb(dummy_model.initialise_var_proc)
txt += get_func_blurb(_get_cos_curve_at_x)
txt += get_func_blurb(dummy_model.fit_process_curve)
txt += "\n\n## Renewal model\n"
txt += get_func_blurb(dummy_model.renew)
dummy_dist = GammaDens()
txt += get_func_blurb(dummy_dist.get_params) + "\n\n"
txt += get_func_blurb(dummy_model.renewal_func)
txt += "\n\n## Calibration\n"
txt += get_func_blurb(run_calibration)
txt += get_func_blurb(custom_init)
dummy_calib = StandardCalib(dummy_model, {}, {})
txt += get_func_blurb(dummy_calib.__init__)
txt += get_func_blurb(dummy_calib.sample_calib_params)

In [3]:
display(Markdown(txt))

## Modelled population size
 We used total population size estimated by the World Bank where a population size was available, and used the estimate provided by the United Nations Department of Economic and Social Affairs otherwise.  Population data were downloaded from [the World Bank](https://databank.worldbank.org/source/population-estimates-and-projections#) on 1 April 2025. The population size for 2020 was used for all countries except for Australia, for which the population size in 2022 was used (because of the later analysis period for this country).  [UN DESA population data](https://population.un.org/wpp/assets/Excel%20Files/1_Indicator%20(Standard)/EXCEL_FILES/2_Population/WPP2024_POP_F02_1_POPULATION_5-YEAR_AGE_GROUPS_BOTH_SEXES.xlsx) was downloaded on 18 March 2025 and the population data needed for 2020 used where necessary. 

## Analysis period
 For all countries but Australia, the start of the calibration period was set to be the time at which the per capita daily rate of deaths passed 2 deaths per million population. However, if this threshold was not reached by 1 June 2020, the simulation commenced at this default time instead. For Australia, the simulation commenced from the time that vaccination reached 90% of its final value.  We substituted Germany for Switzerland and Ireland because these two countries had almost identical profiles of vaccine doses administered per person in the early phases of the roll-out and vaccination data were not available for these two countries during this period. Similarly, we substituted Great Britain for Qatar based on the same rationale. 

 For all countries but Australia, the end time for the analysis was calculated as the time that the population vaccination coverage passed 5%, provided that the vaccination coverage did reach this value before the default end time of 1 December 2021. Otherwise, this default end date was used instead. For Australia, the latest date for which the Google mobility data was available was used. 

## Calibration targets


### WHO indicators
 The number of deaths by week reported by WHO was used as the first calibration target for all countries. Any values of zero in this series were replaced with a value of 0.5 to enable comparison to modelled outputs on the log scale. Deaths was the one of two indicators for which a common dispersion parameter was used for the distribution comparison of the modelled value. The value for weighting the deaths indicator time series was set to 20.  The number of cases by week reported by WHO was used as the second calibration target for all countries. Linear interpolation was used to replace missing values, and as for deaths, any zero values were replaced with a value of 0.5. We only calibrated to cases from the 1 June 2020 onward, because we considered that prior to this time many countries were still rapidly scaling up testing capacity and this indicator may have been less reliable. Cases was the second indicator for which a common dispersion parameter was applied. A target weight was applied to the series of cases such that the weight for each case observation point was the same as for each death observation. 

### Hospitalisations
 A single hospitalisation indicator used for calibration was chosen using a hierarchical approach. In selecting the indicator, the number of new admissions was preferred over estimates of total bed occupancy, and total hospital indicators were preferred over ICU indicators. The final hierarchy of indicators was: 

1. New hospital admissions 

2. Hospital occupancy 

3. New ICU admissions 

4. ICU occupancy 

5. No hospital indicator 



 That is, the highest ranked indicator was used based on data availability, and no hospital indicator was incorporated if none were available.  This indicator was the final calibration target for which a common dispersion parameter was applied. As for cases, a weight was applied to the hospitalisation series such that the weight for each observation point was the same as for each death observation. 



### Seroprevalence
 Seroprevalence data was obtained from [SeroTracker](https://github.com/serotracker/sars-cov-2-data/raw/refs/heads/main/serotracker_dataset.csv) on 11 December 2024, with the date for each serosurvey estimate calculated as the mid-point between the reported start and end dates of sampling. This date was then lagged earlier by 14 for the purposes of calibration to allow for a delay between infection and the subsequent development of detectable antibodies.  Country income classifications were obtained from [the World Bank](https://datacatalogapi.worldbank.org/ddhxext/ResourceDownload?resource_unique_id=DR0090755).  We filtered the SeroTracker data to include only the estimate reported as primary from Unity-aligned national-level surveys for which the number of participants was at least 600. We also considered only a maximum of one seroprevalence value for any given date (keeping the largest estimate of three surveys done on the same day for Mexico).  For any consecutive estimates for which a lower estimate followed an immediately preceding higher estimate, we pooled these two estimates and placed the pooled estimate at the mid-point of the dates of the two estimates. We repeatedly applied this process until seroprevalence estimates were monotonically increasing over time. 

 We compared the modelled proportion ever infected against the seroprevalence reported at least six months (183 days) after the start of the simulation, because a comparison against early seroprevalence estimates would not account for waves of transmission prior to the start of the simulation. We discarded seroprevalence estimates that were less than 5% away from a value of zero or 100%. We also ignored seroprevalence estimates from low and lower middle income countries of Africa, because we were unable to obtain good fits for several of these countries while also maintaining plausible detection/severity parameters (i.e. case detection rate, hospital admission rate and infection fatality rate). That is, we applied much lower priors for these parameters in these countries, although the modelled attack rate still remained well below seroprevalence estimates for some countries. Last, we ignored seroprevalence estimates for Australia, for which the analysis was run largely through 2022, during which time seroprevalence values were much higher. For countries for which seroprevalence calibration targets were available, we assigned a target weight to this indicator of 5 (which is an arbitrary quantity, but can be interpreted with reference to the deaths indicator weight of 20). 



### Variants

#### Data extraction
 Reports of the number of isolates of specific variants of SARS-CoV-2 were obtained from the [covariants](https://github.com/hodcroftlab/covariants/raw/refs/heads/master/cluster_tables/) GitHub repository. Each variant-specific file was downloaded used to create country-specific tables of the variant-specific counts by date.  The identifiers used to identify variants prior to Alpha were: 20A.EU1, 20A.EU2, 20B.S.732A, 21C.Epsilon. The text "Delta" was used to identify Delta variants and the text 21L.Omicron was used to distinguish the BA.2 variant from later variants (modelled as BA.5).  Variant data was considered for dates on which at least 5 sequences were available for the country considered. Further, we required at least 5 such dates be available for that country's variant data to be used as a calibration target. 

 To obtain data to use as calibration targets for both the Alpha and the Delta variants, we used the totals for the country analysed where available. If data were not available for the country, we used pooled data from all the other countries from the same continent where available. 

#### Alpha
 For countries of all continents other than those in Oceania (Australia only) and Africa, a target for the Alpha variant was included in our calibration algorithm. Calibration against data for Alpha started from the beginning of the simulation period (from 1 January 2020). The periods for calibration against the Alpha and the Delta variants were set so as to be mutually exclusive in time. Specifically, the date to transition from calibrating against available data for the Alpha to calibrating against data for Delta was set as 1 March 2021. Exceptions were made for several Asian countries for which this transition date was set one month earlier and two countries of North America for which it was set six weeks later. (The exceptions were Afghanistan: 1 February 2021, India: 1 February 2021, Nepal: 1 December 2020, Indonesia: 1 February 2021, Saudi Arabia: 1 February 2021, Oman: 1 February 2021, Kuwait: 1 February 2021, South Korea: 1 February 2021, Malaysia: 1 February 2021, Honduras: 15 April 2021, Puerto Rico: 15 April 2021.) If this date occurred after the end of the simulation, the Alpha calibration period continued to the end of the simulation. As with the other variants and for the seroprevalence target, decreasing values for the proportion of sequences attributable to Alpha were recursively pooled to ensure they were strictly increasing. The target weight for the Alpha target was set to be 5. 

#### Delta
 For all countries other than Australia, the Delta variant was included if the end date of the calibration fell later than 1 May 2021. Values were again pooled to ensure they were strictly increasing. The target weight for calibration to Delta was set to be 5 for most countries. Exceptions were made if the target for data emerged towards the very end of the calibration last (25 days), in which case a higher weight (of 25) was needed to capture this late emergence with fewer data points. 

#### Omicron BA.2
 The BA.2 calibration target for Australia used the data available from 1 January 2022 to 15 April 2022.  A calibration target for Omicron BA.2 was only included for Oceania, i.e. only Australia. As for other variants, the target weight for BA.2 was set to be 5. 

#### Omicron BA.5
 The BA.5 calibration target for Australia used the data available from 1 April 2022 to 1 September 2022.  As for BA.2, a calibration target for Omicron BA.5 was only included for Oceania. As for other variants, the target weight for BA.5 was set to be 5. 

## Mobility
 For each country, we ran one analysis with no mobility scaling to the transmission rate. We ran one analysis in which Google mobility was used to scale the transmission rate, if mobility data was available from Google. We also ran two analyses in which Facebook mobility was used to scale the transmission rate, if mobility data was available from Facebook. Although Apple mobility data was available and we were able to run analyses using this data source, Apple's terms of use indicate that this source of data cannot be used for this purpose. We contacted Apple, who declined to allow their data to be used for this analysis. For all mobility sources, we smoothed the raw data using a 7 day centred rolling average. For all analyses incorporating mobility scaling, we used an exponential scaling parameter (described in more detail below) which was assigned a uniform prior over limits 0 to 2. 

### No mobility
 For the analysis without mobility, no empiric data was used to scale the transmission rate over time (which was implemented by setting the mobility scaling to one throughout the simulation). 

### Google mobility
 We obtained Google mobility date from [the Google Community Mobility Reports](https://www.gstatic.com/covid19/mobility/Region_Mobility_Report_CSVs.zip) on 14 January 2025 and extracted national mobility estimates by Google mobility domain.  For the Google mobility analysis, we used the raw value interpreted as a percentage plus one to scale the transmission rate.  This approach to incorporating mobility involves a set of priors weighting each mobility domain and one further prior governing the overall influence of the weighted mobility estimate on transmission.  The weights for each mobility domain were normalised to sum to one, with the resulting weighted mobility profile exponentiated to the value specified by the mobility exponential parameter. 

### Facebook mobility
 We used the `all_day_bing_tiles_visited_relative_change` for the first Facebook mobility analysis, and scaled transmission transmission according to one plus this mobility metric.  For the second Facebook mobility analysis, we used the `all_day_ratio_single_tile_users` estimate and scaled transmission according to one minus this mobility metric.  One prior value was incorporated with each of these approaches, which specifies the exponent parameter for the mobility data. This mobility approach was used for each of the two analyses that incorporated Facebook data, using both the tiles visited and the within tile estimates.  The single mobility field is exponentiated to a value specified by a single mobility exponent parameter. 

## Variable process
 Each time point for fitting the variable process was set at intervals through the analysis period spaced by 7 days working backwards from the end of the analysis period. The variable process was then fit to these points using piecewise cosine functions.  The cosine function was obtained by translating and scaling a half cosine function (i.e. a cosine function on domain zero to $\pi$).  The starting value for the variable process was explored as a calibration parameter, along with the subsequent values of the process. This exploration was performed in log space, with the calibrated values for these quantities exponentiated before being used to scale the transmission rate. Each parameter pertaining the variable process was assigned the same prior centred at zero (i.e. no update), and was interpreted as the change in the log-transformed variable process relative to the previous value. 

## Renewal model
 The new strain-specific seeding value was added to the most recent value for the strain-specific history of incidence. This updated strain-specific incidence array was then convolved with the generation interval distribution vector to create a vector of the effective number of infectious individuals. These values were then multiplied by the scalar values representing the variable process, the mobility scaling value and the relative infectiousness of each strain, and divided through by the population size to derive the calculated per capita rate of infection. We then determined the actual rate of infection for each strain as $1 - e^{-r}$ (where $r$ represents the calculated infection rate) to ensure that the per capita rate of infection could not exceed one. These rates of infection were then applied to each possible past history of preceding exposure to variants and their associated susceptibility to infection to calculate the number of people infected from each category and transition them to their new states. 

 For all analyses, the starting population minus the seeding values for the first strain was assigned to the fully susceptible category. Each newly emerging strain was seeded using a triangular pulse of new infections that peaked according to the per capita seeding rate specified. Infectiousness of each variant was calculated with reference to the first modelled variant strain. Persons who had never previously been infected with any strain were considered fully susceptible. We considered partial cross immunity was provided by infection with a preceding variant strain to infection with subsequent strains. However, previously infected persons were assumed to derive complete immunity to the infecting strain and to variant strains that emerged prior to the infecting strain (for example, past infection with Delta conferred complete immunity against future infection with Alpha). A gamma-distributed generation interval was used for the renewal process. The generation interval was truncated at 50 days, because the density reached negligible values beyond this point.  The parameters to each gamma distribution used in our anlaysis were parameterised by analytically calculating the "a" and "scale" parameters from the epidemiologically determined mean and standard deviation. 

 Once the renewal process was run to calculate incidence, the other epidemiological outputs were calculated using a series of convolution operations. Total incidence was first calculated by summing over strain-specific incidence. Cases were then calculated by convolving incidence with a gamma-distributed set of delays from incidence to notification which was truncated after 50 days. This was then multiplied through by the case detection rate (a proportion) and weekly cases were calculated by summing over the preceding 7 days. Deaths were similarly calculated from incidence, but with separate parameters governing the time from incidence to death and with the fraction of incident episodes resulting in death estimated according to the infection fatality rate. For one country (Australia), a reduction in the risk of death was applied because this analysis was performed after wide-scale population vaccination. The relative reduction in the risk of death was set at 0.8 and was not varied during calibration because this would have been collinear with the risk of hospitalisation parameter. As for cases and deaths, hospitalisations were estimated through a convolution distribution with its own parameters and a hospital admission fraction. As for the approach to deaths for Australia, a reduction in the risk of hospitalisation was applied to account for vaccination. The relative reduction in the risk of hospitalisation was set at 0.6 As for cases, deaths and hospitalisations, the convolution of time to ICU admission was parameterised independently, but the proportion of cases resulting in ICU admission was estimated according to the product of the hospital admission fraction and the proportion of hospital admissions resulting in ICU admission. Hospital and ICU occupancy were obtained by convolving the time series of hospital and ICU admissions with the complement of the cumulative distribution of the time to hospital or ICU discharge. Seropositivity was calculated as the proportion of the population remaining in the entirely infection-naive immunity sub-population. Finally, the proportion of incidence attributable to each variant strain was also calculated from the strain-specific incidence. 

## Calibration
 We ran each calibration over 8 chains for 1000 iterations each.  To initialise the model parameters, we started all the updates to the variable process from a value of zero (in logarithmic space) to represent no update, such that the initialisation commenced with the variable process being constant over time. For all other parameters, we used `numpyro`'s `init_to_uniform` method, with a radius of 0.1.  For all epidemiological parameters, the priors were set as described in the parameters section. The dispersion parameter for the variable process was set to a half normal distribution with standard deviation 0.5.  The calibration process calibrates parameters for each update point of the variable process in logarithmic space. The prior distribution for the update for each period of the variable process is a normal distribution centred at a value of zero to represent no change from the previous value. The standard deviation of each normal distribution is provided by the (single) dispersion parameter of the variable process introduced above. 